# AAAI-27 research paper: "Privacy-Preserving Federated Learning for Clinical Text and Records Under Realistic Heterogeneity"
---
## 📄 Part 1: Hierarchical Multi-Label Clinical Text Classification (HMLTC) Modality Codebase

**Modality Developers:** NLP Core Subgroup (Shreya, Hussain & Arpit)  
**Orchestration Hand-off:** Federated Learning Team (Ratan & Harshit)  

---
### Abstract / Context
Clinical text classification requires assigning diagnoses from hierarchies (such as ICD-9/10 CM chapters). Under realistic client heterogeneity, hospital sites encounter severe non-IID skew, label imbalances, and differential privacy constraints. 

This notebook represents the clean, unified, and modular implementation of the NLP Modality. It contains all preprocessing mechanisms, PubMedBERT sequence models, hierarchical consistency losses, and the public handoff API (`FederatedNLPClient`) to allow the FL team to perform global aggregation (e.g. FedAvg, FedProx, SCAFFOLD) without modifying the internal text classification logic.

---

### ⚙️ Section 1 — Centralized Configuration Management

This module centralizes all configuration parameters in `PipelineConfig`. Hyperparameters are cleanly separated into config groups rather than hardcoded in individual classes. The configuration supports:
- Preprocessing overlap sizes and minimum text limits.
- Transformer backbones (defaulting to **PubMedBERT**).
- Multi-label losses (`combined`, `focal`, `hierarchical`, `bce`).
- Local training limits (batch size, learning rates, gradient accumulation steps).

| Parameter Group | Keys | Purpose |
|---|---|---|
| **Data & Paths** | `data_dir`, `output_dir` | Storage locations |
| **Chunking** | `chunk_size`, `chunk_overlap`, `min_chunk_size` | Word-level sliding window controls |
| **Hierarchy** | `coding_system`, `hierarchy_depth` | Dynamic grouping configuration |
| **Encoder Arch** | `model_name`, `max_seq_length`, `use_label_attention` | Backbone configuration |
| **Objective** | `loss_type`, `focal_gamma`, `hierarchy_penalty_weight` | Multi-label losses |
| **Optimizers** | `learning_rate`, `grad_accum_steps`, `max_grad_norm` | Training controls |


In [ ]:
"""
Centralized configuration for the NLP pipeline.

Merges parameters from both Phase 1 (data generation) and Phase 2 (training)
notebooks into a single dataclass-based config.  Every module receives its
parameters from this config; no hardcoded hyperparameters elsewhere.
"""

from __future__ import annotations

import torch
from dataclasses import dataclass, field
from pathlib import Path
from typing import Optional, Tuple


@dataclass
class PipelineConfig:
    """
    Centralized, immutable-by-convention configuration.

    Sections
    --------
    - Data paths & ingestion
    - Text preprocessing / chunking  (kept from Phase 1)
    - Coding-system hierarchy
    - Tokenizer / model architecture
    - Loss function
    - Training hyperparameters
    - Runtime
    """

    # ── Data paths ──────────────────────────────────────────────────────
    data_dir: Path = Path("data_generation")
    output_dir: Path = Path("model_outputs")

    # ── Text preprocessing / chunking (from Phase 1 notebook) ──────────
    chunk_size: int = 512           # target words per chunk
    chunk_overlap: int = 64         # overlap in words
    min_chunk_size: int = 50        # discard chunks smaller than this

    # ── Coding-system hierarchy ─────────────────────────────────────────
    #    System-agnostic: the hierarchy module interprets these at runtime.
    coding_system: str = "auto"     # "icd9", "icd10", "custom", or "auto"
    hierarchy_depth: int = -1       # -1 = unlimited, 2 = two-level, etc.
    hierarchy_file: Optional[str] = None  # optional JSON defining custom hierarchy

    # ── Tokenizer / model ───────────────────────────────────────────────
    model_name: str = (
        "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext"
    )
    max_seq_length: int = 512
    dropout_rate: float = 0.3
    use_label_attention: bool = False  # toggle LabelAttention head (V3)

    # ── Loss function ───────────────────────────────────────────────────
    #    Choices: "bce", "focal", "hierarchical", "combined"
    loss_type: str = "combined"
    focal_gamma: float = 2.0
    focal_alpha: Optional[float] = None  # None = no alpha weighting
    hierarchy_penalty_weight: float = 0.5
    label_smoothing: float = 0.1

    # ── Training hyperparameters ────────────────────────────────────────
    batch_size: int = 8
    learning_rate: float = 2e-5
    num_epochs: int = 5
    warmup_ratio: float = 0.1
    weight_decay: float = 0.01
    grad_accum_steps: int = 2
    max_grad_norm: float = 1.0
    classification_threshold: float = 0.5

    # ── Dataset splits ──────────────────────────────────────────────────
    train_ratio: float = 0.70
    val_ratio: float = 0.15
    test_ratio: float = 0.15

    # ── Runtime ─────────────────────────────────────────────────────────
    random_seed: int = 42
    device: str = field(default_factory=lambda: "cuda" if torch.cuda.is_available() else "cpu")
    num_workers: int = 0
    pin_memory: bool = True
    mixed_precision: bool = False
    log_every_n_steps: int = 10

    # ── Derived properties ──────────────────────────────────────────────

    @property
    def effective_batch_size(self) -> int:
        return self.batch_size * self.grad_accum_steps

    def resolve_paths(self) -> None:
        """Ensure output directories exist."""
        self.data_dir = Path(self.data_dir)
        self.output_dir = Path(self.output_dir)
        self.output_dir.mkdir(parents=True, exist_ok=True)

    def summary(self) -> str:
        """Human-readable config summary."""
        lines = [
            "PipelineConfig",
            f"  model           : {self.model_name}",
            f"  device          : {self.device}",
            f"  loss            : {self.loss_type}",
            f"  coding_system   : {self.coding_system}",
            f"  hierarchy_depth : {self.hierarchy_depth}",
            f"  batch_size      : {self.batch_size} (effective {self.effective_batch_size})",
            f"  lr              : {self.learning_rate}",
            f"  epochs          : {self.num_epochs}",
            f"  max_seq_length  : {self.max_seq_length}",
            f"  dropout         : {self.dropout_rate}",
            f"  label_attention : {self.use_label_attention}",
        ]
        return "\n".join(lines)


### 🛠️ Section 2 — Shared Helper Utilities

Implements JSON reading/writing wrappers and a deterministic, SHA-256-based chunk identifier hashing scheme to generate immutable segment IDs for client logging, tracking, and metadata alignment.

In [ ]:
"""
Shared utility functions.

Kept verbatim from Phase 1 & Phase 2 notebooks:
  - save_json / load_json
  - timestamp
  - generate_chunk_id
"""

from __future__ import annotations

import hashlib
import json
import os
from datetime import datetime
from pathlib import Path
from typing import Any


def save_json(data: Any, filepath: str | Path, silent: bool = False) -> None:
    """Save *data* as a formatted JSON file."""
    filepath = Path(filepath)
    filepath.parent.mkdir(parents=True, exist_ok=True)
    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
    if not silent:
        size_kb = os.path.getsize(filepath) / 1024
        print(f"  ✓ Saved: {filepath}  ({size_kb:.1f} KB)")


def load_json(filepath: str | Path) -> Any:
    """Load a JSON file and return the parsed data."""
    with open(filepath, "r", encoding="utf-8") as f:
        return json.load(f)


def timestamp() -> str:
    """Current ISO timestamp."""
    return datetime.now().isoformat()


def generate_chunk_id(*args: Any) -> str:
    """Generate a deterministic short ID from input arguments."""
    content = "|".join(str(a) for a in args)
    return hashlib.sha256(content.encode("utf-8")).hexdigest()[:16]


### 🧬 Section 3 — Reproducibility Seed Utility

Enforces system-wide determinism across CPU/GPU contexts. It wraps `random`, `numpy`, and `torch` seeding mechanisms, explicitly disabling CuDNN non-deterministic benchmarking for strict experiment repeatability.

In [ ]:
"""
Reproducibility seed utility.

Kept from Phase 2 notebook ``set_seed()``.
"""

from __future__ import annotations

import random

import numpy as np
import torch


def set_seed(seed: int = 42) -> None:
    """Set random seeds for Python, NumPy, and PyTorch for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        # Deterministic behaviour (may hurt perf; toggle if needed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


### 💾 Section 4 — Model Checkpointing Save/Load

Manages client model checkpoints. Saves optimizer and scheduler momentum buffers along with metrics to support mid-training resumes, state-transfers, or best-model rollback based on validation F1.

In [ ]:
"""
Model checkpoint save / load utilities.
"""

from __future__ import annotations

from pathlib import Path
from typing import Any, Dict, Optional

import torch


def save_checkpoint(
    model: torch.nn.Module,
    optimizer: Optional[torch.optim.Optimizer],
    epoch: int,
    metrics: Dict[str, Any],
    filepath: str | Path,
    *,
    scheduler: Optional[Any] = None,
) -> None:
    """
    Save a training checkpoint.

    Parameters
    ----------
    model : nn.Module
    optimizer : Optimizer or None
    epoch : int
    metrics : dict   – validation metrics at this checkpoint
    filepath : Path
    scheduler : optional LR scheduler
    """
    filepath = Path(filepath)
    filepath.parent.mkdir(parents=True, exist_ok=True)

    payload: Dict[str, Any] = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "metrics": metrics,
    }
    if optimizer is not None:
        payload["optimizer_state_dict"] = optimizer.state_dict()
    if scheduler is not None:
        payload["scheduler_state_dict"] = scheduler.state_dict()

    torch.save(payload, filepath)
    print(f"  ✓ Checkpoint saved: {filepath}")


def load_checkpoint(
    filepath: str | Path,
    model: torch.nn.Module,
    optimizer: Optional[torch.optim.Optimizer] = None,
    scheduler: Optional[Any] = None,
    map_location: Optional[str] = None,
) -> Dict[str, Any]:
    """
    Restore a training checkpoint.

    Returns the ``metrics`` dict stored in the checkpoint so the caller
    can decide whether to keep this as best.
    """
    filepath = Path(filepath)
    ckpt = torch.load(filepath, map_location=map_location, weights_only=False)

    model.load_state_dict(ckpt["model_state_dict"])

    if optimizer is not None and "optimizer_state_dict" in ckpt:
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])

    if scheduler is not None and "scheduler_state_dict" in ckpt:
        scheduler.load_state_dict(ckpt["scheduler_state_dict"])

    print(f"  ✓ Checkpoint loaded: {filepath}  (epoch {ckpt.get('epoch', '?')})")
    return ckpt


def find_best_checkpoint(directory: str | Path, metric_key: str = "val_f1_micro") -> Optional[Path]:
    """
    Scan *directory* for ``*.pt`` checkpoints and return the path to the
    one with the highest value of *metric_key*.
    """
    directory = Path(directory)
    best_path = None
    best_score = -1.0

    for ckpt_path in sorted(directory.glob("*.pt")):
        try:
            ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
            score = ckpt.get("metrics", {}).get(metric_key, -1.0)
            if score > best_score:
                best_score = score
                best_path = ckpt_path
        except Exception:
            continue

    return best_path


### 📝 Section 5 — Preprocessing & Overlapping Chunking Windows

Implements clinical-text cleansing and sliding window splitting. Key enhancements:
- **MIMIC Redaction Stripping**: Filters patient de-identification tags matching regex `\[\*\*[^\*]*\*\*\]` to prevent sequence pollution.
- **Header Case Normalization**: Identifies common clinical section titles and normalizes formatting.
- **Paragraph-Aware Sliding Window**: Breaks text at paragraph breaks to preserve semantic boundaries. Uses sliding overlap windows:

$$\text{Overlap Window} = \text{chunk\_overlap\_words}$$

In [ ]:
"""
Text preprocessing utilities.

**Kept verbatim** from Phase 1 notebook (``01_Dataset_Generation.ipynb``):
  - ``clean_text``          – unicode normalisation, whitespace collapse
  - ``split_into_paragraphs`` – double-newline splitting
  - ``chunk_paragraphs``    – paragraph-aware sliding-window chunker
  - ``word_count``

**Added** for clinical text:
  - ``clean_clinical_text`` – extends ``clean_text`` with MIMIC-style
    de-identification pattern removal and section-header normalisation.
"""

from __future__ import annotations

import re
from typing import List


# ─────────────────────────────────────────────────────────────────────
# Kept verbatim from Phase 1 notebook
# ─────────────────────────────────────────────────────────────────────

def word_count(text: str) -> int:
    """Count words in *text*."""
    return len(text.split())


def clean_text(text: str) -> str:
    """
    Clean extracted text: normalise whitespace, remove control characters,
    and fix common encoding artefacts.

    Kept from Phase 1 notebook ``clean_text()``.
    """
    if not text:
        return ""
    # Remove form feeds and null bytes
    text = text.replace("\x0c", " ").replace("\x00", "")
    # Normalise various dash types to standard hyphen
    text = re.sub(r"[\u2013\u2014\u2015]", "-", text)
    # Normalise quotes
    text = re.sub(r"[\u201c\u201d\u201e\u201f]", '"', text)
    text = re.sub(r"[\u2018\u2019\u201a\u201b]", "'", text)
    # Collapse multiple whitespace (but preserve paragraph breaks)
    text = re.sub(r"[^\S\n]+", " ", text)       # spaces/tabs to single space
    text = re.sub(r"\n{3,}", "\n\n", text)       # 3+ newlines to 2
    # Remove lines that are just whitespace
    lines = [line.strip() for line in text.split("\n")]
    text = "\n".join(lines)
    return text.strip()


def split_into_paragraphs(text: str) -> List[str]:
    """
    Split *text* into paragraphs by double-newline boundaries.
    Returns non-empty paragraph strings.

    Kept from Phase 1 notebook ``split_into_paragraphs()``.
    """
    raw_paragraphs = re.split(r"\n\s*\n", text)
    return [p.strip() for p in raw_paragraphs if p.strip()]


def chunk_paragraphs(
    paragraphs: List[str],
    target_words: int = 512,
    overlap_words: int = 64,
    min_words: int = 50,
) -> List[str]:
    """
    Group paragraphs into chunks targeting *target_words* word count.
    Preserves paragraph boundaries.  Uses word-count-based overlap.

    Kept from Phase 1 notebook ``chunk_paragraphs()``.
    """
    chunks: List[str] = []
    current_chunk_paras: List[str] = []
    current_word_count = 0

    for para in paragraphs:
        para_wc = word_count(para)

        # If adding this paragraph exceeds the target and we have content
        if current_word_count + para_wc > target_words and current_chunk_paras:
            # Save current chunk
            chunk_text = "\n\n".join(current_chunk_paras)
            if word_count(chunk_text) >= min_words:
                chunks.append(chunk_text)

            # Start new chunk with overlap
            overlap_paras: List[str] = []
            overlap_wc = 0
            for p in reversed(current_chunk_paras):
                p_wc = word_count(p)
                if overlap_wc + p_wc > overlap_words and overlap_paras:
                    break
                overlap_paras.insert(0, p)
                overlap_wc += p_wc

            current_chunk_paras = overlap_paras
            current_word_count = overlap_wc

        current_chunk_paras.append(para)
        current_word_count += para_wc

    # Final chunk
    if current_chunk_paras:
        chunk_text = "\n\n".join(current_chunk_paras)
        if word_count(chunk_text) >= min_words:
            chunks.append(chunk_text)

    return chunks


# ─────────────────────────────────────────────────────────────────────
# Added for clinical text (MIMIC compatibility)
# ─────────────────────────────────────────────────────────────────────

# MIMIC de-identification pattern: [**First Name**], [**2100-01-01**], etc.
_MIMIC_DEIDENT_RE = re.compile(r"\[\*\*[^\]]*\*\*\]")

# Common clinical section headers (normalise casing)
_SECTION_HEADER_RE = re.compile(
    r"^(HISTORY OF PRESENT ILLNESS|DISCHARGE DIAGNOSIS|"
    r"DISCHARGE MEDICATIONS|HOSPITAL COURSE|"
    r"CHIEF COMPLAINT|PAST MEDICAL HISTORY|"
    r"SOCIAL HISTORY|FAMILY HISTORY|"
    r"PHYSICAL EXAMINATION|LABORATORY DATA|"
    r"DISCHARGE INSTRUCTIONS|DISCHARGE CONDITION|"
    r"ADMISSION DATE|DISCHARGE DATE|"
    r"ALLERGIES|MEDICATIONS ON ADMISSION)\s*:?\s*",
    re.IGNORECASE | re.MULTILINE,
)


def clean_clinical_text(text: str, *, remove_deident: bool = True) -> str:
    """
    Extend ``clean_text`` with clinical-note-specific cleaning:

    1. Remove MIMIC-style de-identification brackets ``[**...**]``.
    2. Normalise common clinical section headers.
    3. Apply all standard ``clean_text`` transformations.

    Parameters
    ----------
    text : str
        Raw clinical note.
    remove_deident : bool
        If *True*, replace ``[**...**]`` placeholders with a single space.
    """
    if not text:
        return ""

    if remove_deident:
        text = _MIMIC_DEIDENT_RE.sub(" ", text)

    # Normalise section headers to Title Case with colon
    def _normalise_header(m: re.Match) -> str:
        return m.group(1).strip().title() + ":\n"

    text = _SECTION_HEADER_RE.sub(_normalise_header, text)

    # Delegate the rest to the original clean_text
    return clean_text(text)


def chunk_clinical_text(
    text: str,
    target_words: int = 512,
    overlap_words: int = 64,
    min_words: int = 50,
) -> List[str]:
    """
    Convenience: clean → split paragraphs → chunk.

    Parameters
    ----------
    text : str
        Raw or lightly cleaned clinical note.

    Returns
    -------
    List[str]
        List of text chunks.
    """
    cleaned = clean_clinical_text(text)
    paragraphs = split_into_paragraphs(cleaned)
    return chunk_paragraphs(paragraphs, target_words, overlap_words, min_words)


### 🗂️ Section 6 — Coding-System Agnostic Hierarchy Builder

To adapt from ADNI taxonomy to MIMIC-style ICD hierarchies, this module implements a **coding-system agnostic** parent-child mapper. It builds direct associations from dynamic lists of codes:
- Fully configurable parent levels and label depths (e.g. 3-digit ICD truncate codes).
- Enforces parent-child hierarchy consistency:

$$\forall \text{ child } c, \text{ parent } p = \text{Parent}(c) \implies y_c \le y_p$$

- Standardizes mapping structures for cross-client federated label consistency.

In [ ]:
"""
Coding-system-agnostic hierarchical label structure.

Designed to work with **any** medical coding system (ICD-9-CM, ICD-10-CM,
CPT, SNOMED-CT, or a custom taxonomy).  The hierarchy depth is fully
configurable — it is **not** fixed at two levels.

The module ships with built-in chapter tables for ICD-9 and ICD-10 to
bootstrap hierarchy construction when those systems are detected, but the
``CodingSystemHierarchy`` class accepts arbitrary parent→child mappings
through its ``from_mapping`` classmethod.

Usage
-----
>>> h = CodingSystemHierarchy.from_codes(["428.0", "410.9", "250.00"], system="icd9")
>>> h.parent_to_children
{'Diseases Of The Circulatory System': ['428', '410'],
 'Endocrine, Nutritional And Metabolic Diseases': ['250']}
>>> h.child_to_parent["428"]
'Diseases Of The Circulatory System'
"""

from __future__ import annotations

import json
from collections import defaultdict
from pathlib import Path
from typing import Any, Dict, List, Optional, Set, Tuple


# ════════════════════════════════════════════════════════════════════
# Built-in ICD chapter tables
# ════════════════════════════════════════════════════════════════════

# ICD-9-CM chapters — (start, end) ranges for the *numeric* portion of
# the three-digit code.  V/E codes are handled specially.
ICD9_CHAPTERS: List[Tuple[str, int, int]] = [
    ("Infectious And Parasitic Diseases", 1, 139),
    ("Neoplasms", 140, 239),
    ("Endocrine, Nutritional And Metabolic Diseases", 240, 279),
    ("Diseases Of The Blood", 280, 289),
    ("Mental Disorders", 290, 319),
    ("Diseases Of The Nervous System", 320, 389),
    ("Diseases Of The Circulatory System", 390, 459),
    ("Diseases Of The Respiratory System", 460, 519),
    ("Diseases Of The Digestive System", 520, 579),
    ("Diseases Of The Genitourinary System", 580, 629),
    ("Complications Of Pregnancy", 630, 679),
    ("Diseases Of The Skin", 680, 709),
    ("Diseases Of The Musculoskeletal System", 710, 739),
    ("Congenital Anomalies", 740, 759),
    ("Certain Conditions Originating In The Perinatal Period", 760, 779),
    ("Symptoms, Signs, And Ill-Defined Conditions", 780, 799),
    ("Injury And Poisoning", 800, 999),
]

# ICD-10-CM chapters — identified by the first letter of the code.
ICD10_CHAPTERS: Dict[str, str] = {
    "A": "Certain Infectious And Parasitic Diseases",
    "B": "Certain Infectious And Parasitic Diseases",
    "C": "Neoplasms",
    "D": "Diseases Of The Blood And Neoplasms",
    "E": "Endocrine, Nutritional And Metabolic Diseases",
    "F": "Mental And Behavioural Disorders",
    "G": "Diseases Of The Nervous System",
    "H": "Diseases Of The Eye And Ear",
    "I": "Diseases Of The Circulatory System",
    "J": "Diseases Of The Respiratory System",
    "K": "Diseases Of The Digestive System",
    "L": "Diseases Of The Skin",
    "M": "Diseases Of The Musculoskeletal System",
    "N": "Diseases Of The Genitourinary System",
    "O": "Pregnancy, Childbirth And The Puerperium",
    "P": "Certain Conditions Originating In The Perinatal Period",
    "Q": "Congenital Malformations",
    "R": "Symptoms And Signs",
    "S": "Injury And Poisoning",
    "T": "Injury And Poisoning",
    "U": "Codes For Special Purposes",
    "V": "External Causes Of Morbidity",
    "W": "External Causes Of Morbidity",
    "X": "External Causes Of Morbidity",
    "Y": "External Causes Of Morbidity",
    "Z": "Factors Influencing Health Status",
}


# ════════════════════════════════════════════════════════════════════
# Helper: detect coding system from sample codes
# ════════════════════════════════════════════════════════════════════

def detect_coding_system(codes: List[str]) -> str:
    """
    Heuristically decide whether *codes* are ICD-9, ICD-10, or unknown.

    Returns ``"icd9"``, ``"icd10"``, or ``"unknown"``.
    """
    if not codes:
        return "unknown"

    sample = [c.strip() for c in codes[:200]]

    # ICD-10 codes start with a letter followed by digits
    icd10_count = sum(1 for c in sample if len(c) >= 3 and c[0].isalpha() and c[1:3].isdigit())
    # ICD-9 codes are typically 3–5 digit (possibly with leading V/E)
    icd9_count = sum(1 for c in sample if c.replace(".", "").lstrip("VE").isdigit())

    if icd10_count > icd9_count:
        return "icd10"
    if icd9_count > 0:
        return "icd9"
    return "unknown"


# ════════════════════════════════════════════════════════════════════
# Helper: extract ancestor chain at configurable depth
# ════════════════════════════════════════════════════════════════════

def _icd9_code_to_chapter(code: str) -> Optional[str]:
    """Map a single ICD-9 code to its chapter name, or *None*."""
    code = code.strip().upper()
    if code.startswith("V"):
        return "Supplementary Classification V Codes"
    if code.startswith("E"):
        return "Supplementary Classification E Codes"
    # Extract the numeric prefix (first 3 digits)
    numeric = code.replace(".", "")[:3]
    try:
        num = int(numeric)
    except ValueError:
        return None
    for chapter_name, lo, hi in ICD9_CHAPTERS:
        if lo <= num <= hi:
            return chapter_name
    return None


def _icd10_code_to_chapter(code: str) -> Optional[str]:
    """Map a single ICD-10 code to its chapter name, or *None*."""
    code = code.strip().upper()
    if not code:
        return None
    return ICD10_CHAPTERS.get(code[0])


def _truncate_code(code: str, depth: int) -> str:
    """
    Truncate a diagnosis code to *depth* characters (excluding dots).

    For depth=3, ``"428.32"`` → ``"428"`` and ``"E11.65"`` → ``"E11"``.
    """
    stripped = code.replace(".", "")
    truncated = stripped[:depth]
    return truncated


# ════════════════════════════════════════════════════════════════════
# Main class
# ════════════════════════════════════════════════════════════════════

class CodingSystemHierarchy:
    """
    Coding-system-agnostic hierarchical label structure.

    Attributes
    ----------
    parent_to_children : dict[str, list[str]]
        Mapping from parent labels to their child labels.
    child_to_parent : dict[str, str]
        Reverse mapping from each child label to its parent.
    all_labels : list[str]
        Sorted list of every label (parents + children, de-duplicated).
    parent_labels : list[str]
        Sorted list of parent-only labels.
    child_labels : list[str]
        Sorted list of child-only labels.
    depth : int
        Hierarchy depth that was used to build this structure.
    coding_system : str
        Coding system identifier (``"icd9"``, ``"icd10"``, ``"custom"``).
    """

    def __init__(
        self,
        parent_to_children: Dict[str, List[str]],
        coding_system: str = "custom",
        depth: int = -1,
    ) -> None:
        self.coding_system = coding_system
        self.depth = depth

        # Deduplicate children lists while preserving order
        self.parent_to_children: Dict[str, List[str]] = {
            p: list(dict.fromkeys(children))
            for p, children in parent_to_children.items()
        }

        # Build reverse mapping
        self.child_to_parent: Dict[str, str] = {}
        for parent, children in self.parent_to_children.items():
            for child in children:
                self.child_to_parent[child] = parent

        self.parent_labels: List[str] = sorted(self.parent_to_children.keys())
        self.child_labels: List[str] = sorted(self.child_to_parent.keys())
        self.all_labels: List[str] = sorted(
            set(self.parent_labels) | set(self.child_labels)
        )

    # ── Constructors ────────────────────────────────────────────────

    @classmethod
    def from_codes(
        cls,
        codes: List[str],
        system: str = "auto",
        depth: int = -1,
    ) -> "CodingSystemHierarchy":
        """
        Build hierarchy from a flat list of diagnosis codes.

        Parameters
        ----------
        codes : list[str]
            Raw ICD codes (e.g. ``["428.0", "250.01", "410.9"]``).
        system : str
            ``"icd9"``, ``"icd10"``, ``"auto"`` (auto-detect), or
            ``"custom"`` (treat codes as flat labels — no parent grouping).
        depth : int
            Number of characters (excluding dots) to retain when building
            the child-level codes.  ``-1`` means use the full code.
            ``3`` is typical for ICD-9 three-digit grouping.

        Returns
        -------
        CodingSystemHierarchy
        """
        if system == "auto":
            system = detect_coding_system(codes)

        if system == "icd9":
            chapter_fn = _icd9_code_to_chapter
        elif system == "icd10":
            chapter_fn = _icd10_code_to_chapter
        else:
            # Custom / unknown — create a flat hierarchy (every code is its
            # own parent, no children).
            p2c: Dict[str, List[str]] = {c: [] for c in sorted(set(codes))}
            return cls(p2c, coding_system=system, depth=depth)

        # Build parent→children mapping
        parent_to_children: Dict[str, List[str]] = defaultdict(list)
        seen_children: Set[str] = set()

        for raw_code in codes:
            raw_code = raw_code.strip()
            if not raw_code:
                continue

            chapter = chapter_fn(raw_code)
            if chapter is None:
                continue

            # Determine child label (possibly truncated)
            if depth > 0:
                child = _truncate_code(raw_code, depth)
            else:
                child = raw_code.replace(".", "")

            if child not in seen_children:
                parent_to_children[chapter].append(child)
                seen_children.add(child)

        return cls(dict(parent_to_children), coding_system=system, depth=depth)

    @classmethod
    def from_mapping(
        cls,
        mapping: Dict[str, List[str]],
        coding_system: str = "custom",
    ) -> "CodingSystemHierarchy":
        """
        Build from an explicit parent → children dict.

        This is the escape hatch for any taxonomy that doesn't use standard
        ICD codes (e.g. SNOMED-CT groups, or your old ADNI taxonomy).
        """
        return cls(mapping, coding_system=coding_system, depth=-1)

    @classmethod
    def from_json(cls, filepath: str | Path) -> "CodingSystemHierarchy":
        """Load a hierarchy previously saved with ``to_json``."""
        with open(filepath, "r", encoding="utf-8") as f:
            data = json.load(f)
        return cls(
            parent_to_children=data["parent_to_children"],
            coding_system=data.get("coding_system", "custom"),
            depth=data.get("depth", -1),
        )

    # ── Serialisation ───────────────────────────────────────────────

    def to_json(self, filepath: str | Path) -> None:
        """Save hierarchy to JSON for reproducibility across clients."""
        filepath = Path(filepath)
        filepath.parent.mkdir(parents=True, exist_ok=True)
        payload = {
            "coding_system": self.coding_system,
            "depth": self.depth,
            "parent_to_children": self.parent_to_children,
            "child_to_parent": self.child_to_parent,
            "parent_labels": self.parent_labels,
            "child_labels": self.child_labels,
        }
        with open(filepath, "w", encoding="utf-8") as f:
            json.dump(payload, f, indent=2, ensure_ascii=False)
        print(f"  ✓ Hierarchy saved: {filepath}")

    # ── Queries ──────────────────────────────────────────────────────

    def get_parent(self, child_label: str) -> Optional[str]:
        """Return the parent of *child_label*, or *None*."""
        return self.child_to_parent.get(child_label)

    def get_children(self, parent_label: str) -> List[str]:
        """Return children of *parent_label* (empty list if not found)."""
        return self.parent_to_children.get(parent_label, [])

    def ensure_parent_consistency(self, labels: List[str]) -> List[str]:
        """
        Given a list of labels, ensure every child's parent is present.

        Returns a new list with missing parents added.
        """
        label_set = set(labels)
        for label in list(label_set):
            parent = self.child_to_parent.get(label)
            if parent is not None:
                label_set.add(parent)
        return sorted(label_set)

    @property
    def num_parents(self) -> int:
        return len(self.parent_labels)

    @property
    def num_children(self) -> int:
        return len(self.child_labels)

    @property
    def num_labels(self) -> int:
        return len(self.all_labels)

    # ── Display ──────────────────────────────────────────────────────

    def summary(self) -> str:
        lines = [
            f"CodingSystemHierarchy  (system={self.coding_system}, depth={self.depth})",
            f"  Parents  : {self.num_parents}",
            f"  Children : {self.num_children}",
            f"  Total    : {self.num_labels}",
        ]
        for parent in self.parent_labels:
            children = self.parent_to_children[parent]
            lines.append(f"    {parent}  ({len(children)} children)")
            for child in children[:5]:
                lines.append(f"      ├── {child}")
            if len(children) > 5:
                lines.append(f"      └── ... +{len(children) - 5} more")
        return "\n".join(lines)

    def __repr__(self) -> str:
        return (
            f"CodingSystemHierarchy(system={self.coding_system!r}, "
            f"parents={self.num_parents}, children={self.num_children})"
        )


### 🏷️ Section 7 — Hierarchical Multi-Hot Label Encoder

Binarizes multi-label lists into mathematical multi-hot vectors $\mathbf{y} \in \{0, 1\}^{C}$. Wraps `MultiLabelBinarizer` and automatically injects missing parents when a child diagnosis is predicted to ensure hierarchy integrity.

In [ ]:
"""
Hierarchical multi-hot label encoder.

Wraps ``sklearn.preprocessing.MultiLabelBinarizer`` and adds:
  - automatic parent-consistency enforcement
  - save / load for federated reproducibility
  - decode helpers

The multi-hot encoding logic is **kept** from Phase 2 notebook
(``extract_texts_and_labels`` / ``mlb``), but elevated into a reusable class.
"""

from __future__ import annotations

import json
from pathlib import Path
from typing import Dict, List, Optional

import numpy as np
from sklearn.preprocessing import MultiLabelBinarizer

from nlp_pipeline.data.hierarchy import CodingSystemHierarchy


class HierarchicalLabelEncoder:
    """
    Multi-hot encoder that respects a hierarchical label structure.

    Parameters
    ----------
    hierarchy : CodingSystemHierarchy
        The parent↔child label structure.
    enforce_consistency : bool
        If *True*, ``encode`` will auto-add missing parent labels when a
        child is present (same logic as Phase 1 ``validate_labels``).
    """

    def __init__(
        self,
        hierarchy: CodingSystemHierarchy,
        enforce_consistency: bool = True,
    ) -> None:
        self.hierarchy = hierarchy
        self.enforce_consistency = enforce_consistency

        # Deterministic sorted label list — must be identical across all
        # federated clients for the label vectors to align.
        self.label_names: List[str] = list(hierarchy.all_labels)
        self.num_labels: int = len(self.label_names)

        # Index mappings
        self.label_to_idx: Dict[str, int] = {
            label: idx for idx, label in enumerate(self.label_names)
        }
        self.idx_to_label: Dict[int, str] = {
            idx: label for label, idx in self.label_to_idx.items()
        }

        # Fit the sklearn binarizer once
        self._mlb = MultiLabelBinarizer(classes=self.label_names)
        self._mlb.fit([self.label_names])

    # ── Encode / decode ────────────────────────────────────────────

    def encode(self, label_lists: List[List[str]]) -> np.ndarray:
        """
        Encode a batch of label lists into a multi-hot matrix.

        Parameters
        ----------
        label_lists : list[list[str]]
            Each inner list contains label strings for one sample.

        Returns
        -------
        np.ndarray of shape ``(N, num_labels)`` with dtype ``float32``.
        """
        if self.enforce_consistency:
            label_lists = [
                self.hierarchy.ensure_parent_consistency(labels)
                for labels in label_lists
            ]

        return self._mlb.transform(label_lists).astype(np.float32)

    def encode_single(self, labels: List[str]) -> np.ndarray:
        """Encode a single sample's labels → 1-D array."""
        return self.encode([labels])[0]

    def decode(
        self,
        predictions: np.ndarray,
        threshold: float = 0.5,
    ) -> List[List[str]]:
        """
        Decode a probability / binary matrix back to label lists.

        Parameters
        ----------
        predictions : np.ndarray
            Shape ``(N, num_labels)`` — probabilities or binary 0/1.
        threshold : float
            Applied when *predictions* are continuous.
        """
        binary = (predictions >= threshold).astype(int)
        return [list(row) for row in self._mlb.inverse_transform(binary)]

    def decode_single(
        self,
        prediction: np.ndarray,
        threshold: float = 0.5,
    ) -> List[str]:
        """Decode a single 1-D prediction vector."""
        return self.decode(prediction.reshape(1, -1), threshold)[0]

    # ── Serialisation ──────────────────────────────────────────────

    def save(self, filepath: str | Path) -> None:
        """
        Save label mappings to JSON.

        Must be loaded on every federated client to ensure identical
        label ordering.
        """
        filepath = Path(filepath)
        filepath.parent.mkdir(parents=True, exist_ok=True)
        payload = {
            "label_names": self.label_names,
            "label_to_idx": self.label_to_idx,
            "num_labels": self.num_labels,
            "enforce_consistency": self.enforce_consistency,
        }
        with open(filepath, "w", encoding="utf-8") as f:
            json.dump(payload, f, indent=2, ensure_ascii=False)
        print(f"  ✓ Label encoder saved: {filepath}")

    @classmethod
    def load(
        cls,
        filepath: str | Path,
        hierarchy: CodingSystemHierarchy,
    ) -> "HierarchicalLabelEncoder":
        """Restore from a JSON file previously created by ``save``."""
        with open(filepath, "r", encoding="utf-8") as f:
            data = json.load(f)
        encoder = cls(
            hierarchy=hierarchy,
            enforce_consistency=data.get("enforce_consistency", True),
        )
        # Verify that label ordering is consistent
        if encoder.label_names != data["label_names"]:
            raise ValueError(
                "Label name ordering mismatch between saved encoder and "
                "current hierarchy.  This would cause mis-aligned label "
                "vectors across federated clients."
            )
        return encoder

    # ── Display ────────────────────────────────────────────────────

    def __repr__(self) -> str:
        return (
            f"HierarchicalLabelEncoder(num_labels={self.num_labels}, "
            f"consistency={self.enforce_consistency})"
        )


### 🗃️ Section 8 — PyTorch Clinical Text Dataset

A standard PyTorch `Dataset` that takes input text segments, runs HuggingFace tokenization pipelines, aligns masks, and returns multi-hot FloatTensors for the objective layers.

In [ ]:
"""
PyTorch Dataset for hierarchical multi-label clinical text classification.

**Kept** from Phase 2 notebook ``MultiLabelTextDataset``:
  - ``__getitem__`` returns ``{input_ids, attention_mask, labels}``
  - Tokenisation with HuggingFace tokenizer
  - Label tensor conversion

**Modified**:
  - Renamed to ``ClinicalTextDataset``
  - Added ``from_records`` classmethod for standardised record ingestion
  - Accepts optional pre-computed label matrices
"""

from __future__ import annotations

from typing import Dict, List, Optional

import numpy as np
import torch
from torch.utils.data import Dataset
from transformers import PreTrainedTokenizerBase

from nlp_pipeline.data.label_encoder import HierarchicalLabelEncoder


class ClinicalTextDataset(Dataset):
    """
    PyTorch Dataset for multi-label text classification.

    Each sample returns::

        {
            "input_ids":      LongTensor  (max_length,),
            "attention_mask": LongTensor  (max_length,),
            "labels":         FloatTensor (num_labels,),
        }

    Parameters
    ----------
    texts : list[str]
        Raw text strings.
    label_matrix : np.ndarray
        Multi-hot label matrix of shape ``(N, num_labels)``.
    tokenizer : PreTrainedTokenizerBase
        HuggingFace tokenizer (e.g. PubMedBERT).
    max_length : int
        Maximum token sequence length.
    """

    def __init__(
        self,
        texts: List[str],
        label_matrix: np.ndarray,
        tokenizer: PreTrainedTokenizerBase,
        max_length: int = 512,
    ) -> None:
        assert len(texts) == label_matrix.shape[0], (
            f"texts ({len(texts)}) and labels ({label_matrix.shape[0]}) "
            f"must have the same number of samples."
        )
        self.texts = texts
        self.labels = label_matrix.astype(np.float32)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self) -> int:
        return len(self.texts)

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        text = self.texts[idx]
        label = self.labels[idx]

        encoding = self.tokenizer(
            text,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.FloatTensor(label),
        }

    # ── Convenience constructors ────────────────────────────────────

    @classmethod
    def from_records(
        cls,
        records: List[Dict],
        label_encoder: HierarchicalLabelEncoder,
        tokenizer: PreTrainedTokenizerBase,
        max_length: int = 512,
        text_key: str = "text",
        label_key: str = "labels",
    ) -> "ClinicalTextDataset":
        """
        Build a dataset from the standardised record format::

            {"text": "...", "labels": ["LabelA", "LabelB"]}

        This is the format produced by every ``BaseDataPlugin``.
        """
        texts = [r[text_key] for r in records]
        raw_labels = [r[label_key] for r in records]
        label_matrix = label_encoder.encode(raw_labels)
        return cls(texts, label_matrix, tokenizer, max_length)

    def __repr__(self) -> str:
        return (
            f"ClinicalTextDataset(samples={len(self)}, "
            f"num_labels={self.labels.shape[1]}, "
            f"max_length={self.max_length})"
        )


### 🔌 Section 9 — Abstract Data Ingestion Interfaces

Implements a pluggable interface design. Defines `BaseDataPlugin` to decouple raw data storage layout from training code. Includes a concrete `JSONDataPlugin` and a placeholder `MIMICDataPlugin` to easily load MIMIC-IV notes and diagnoses.

In [ ]:
"""
Abstract data ingestion interface (plugin pattern).

Defines ``BaseDataPlugin`` — the contract that every data source must
implement — and ships two concrete plugins:

  - ``JSONDataPlugin``  – loads Phase 1 JSON outputs (backward-compatible)
  - ``MIMICDataPlugin`` – placeholder for MIMIC-IV discharge notes + ICD codes

The NLP pipeline never knows *where* the data comes from; it only sees
standardised records::

    {"patient_id": ..., "admission_id": ..., "text": "...", "labels": [...]}
"""

from __future__ import annotations

import json
from abc import ABC, abstractmethod
from pathlib import Path
from typing import Any, Dict, List, Optional

from nlp_pipeline.data.preprocess import clean_clinical_text


# ════════════════════════════════════════════════════════════════════
# Base class
# ════════════════════════════════════════════════════════════════════

class BaseDataPlugin(ABC):
    """
    Abstract base for all data-ingestion plugins.

    Subclasses must implement ``load_records`` which returns a list of
    standardised record dicts.
    """

    @abstractmethod
    def load_records(self) -> List[Dict[str, Any]]:
        """
        Load and return standardised records.

        Each record **must** contain at least::

            {
                "text":   str,    # clinical note or text chunk
                "labels": list,   # list of label strings
            }

        Optional fields (used for provenance tracking)::

            {
                "patient_id":   str | int,
                "admission_id": str | int,
            }
        """
        ...


# ════════════════════════════════════════════════════════════════════
# Concrete: JSON files from Phase 1
# ════════════════════════════════════════════════════════════════════

class JSONDataPlugin(BaseDataPlugin):
    """
    Load records from Phase 1 JSON outputs (``final_dataset.json``,
    ``train.json``, ``val.json``, ``test.json``).

    This provides **backward compatibility** with the existing notebooks.

    Parameters
    ----------
    filepath : str | Path
        Path to a JSON file containing a list of records.
    text_key : str
        Key for the text field in each record.
    label_key : str
        Key for the labels field in each record.
    clean : bool
        If *True*, apply ``clean_clinical_text`` to each record's text.
    """

    def __init__(
        self,
        filepath: str | Path,
        text_key: str = "text",
        label_key: str = "labels",
        clean: bool = False,
    ) -> None:
        self.filepath = Path(filepath)
        self.text_key = text_key
        self.label_key = label_key
        self.clean = clean

    def load_records(self) -> List[Dict[str, Any]]:
        with open(self.filepath, "r", encoding="utf-8") as f:
            raw = json.load(f)

        records: List[Dict[str, Any]] = []
        for item in raw:
            text = item.get(self.text_key, "")
            if self.clean:
                text = clean_clinical_text(text)

            records.append({
                "text": text,
                "labels": item.get(self.label_key, []),
                "patient_id": item.get("patient_id",
                              item.get("metadata", {}).get("chunk_id", "")),
                "admission_id": item.get("admission_id",
                                item.get("metadata", {}).get("doc_id", "")),
            })

        return records

    def __repr__(self) -> str:
        return f"JSONDataPlugin(filepath={self.filepath})"


# ════════════════════════════════════════════════════════════════════
# Concrete: MIMIC-IV placeholder
# ════════════════════════════════════════════════════════════════════

class MIMICDataPlugin(BaseDataPlugin):
    """
    Placeholder data plugin for MIMIC-IV discharge summaries.

    .. note::

       This plugin will be implemented once PhysioNet access is granted.
       The expected workflow is:

       1. Read ``discharge.csv.gz`` from MIMIC-IV-Note
       2. Read ``diagnoses_icd.csv.gz`` from MIMIC-IV
       3. Merge on ``(subject_id, hadm_id)``
       4. Clean notes with ``clean_clinical_text``
       5. Return standardised records

    Parameters
    ----------
    mimic_dir : str | Path
        Root directory of the MIMIC-IV dataset.
    note_table : str
        Filename / table name for the notes (default: ``discharge``).
    diag_table : str
        Filename / table name for diagnoses (default: ``diagnoses_icd``).
    """

    def __init__(
        self,
        mimic_dir: str | Path,
        note_table: str = "discharge",
        diag_table: str = "diagnoses_icd",
    ) -> None:
        self.mimic_dir = Path(mimic_dir)
        self.note_table = note_table
        self.diag_table = diag_table

    def load_records(self) -> List[Dict[str, Any]]:
        raise NotImplementedError(
            "MIMICDataPlugin.load_records() is not yet implemented.  "
            "Provide PhysioNet-credentialed MIMIC-IV files in "
            f"'{self.mimic_dir}' and implement this method.  "
            "Expected tables: "
            f"'{self.note_table}', '{self.diag_table}'."
        )

    def __repr__(self) -> str:
        return f"MIMICDataPlugin(mimic_dir={self.mimic_dir})"


### 🤖 Section 10 — PubMedBERT Classifier Architecture

Defines `ClinicalHMLTCModel` using **PubMedBERT** (`microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext`).
- **CLS contextual pooling**: Extracts sequence representation from $[CLS]$ token outputs.
- **Label-Aware Attention (Optional)**: Preserves the V3 attention head. Learns a query vector per label to weigh token sequences:

$$\alpha_{ij} = \text{softmax}\left(\frac{\mathbf{q}_i \mathbf{h}_j^T}{\sqrt{d}}\right)$$

$$\mathbf{r}_i = \sum_j \alpha_{ij} \mathbf{h}_j$$

Outputs raw logits; no loss calculations or sigmoids are hardcoded inside the model.

In [ ]:
"""
PubMedBERT-based Hierarchical Multi-Label Clinical Text Classifier.

**Kept** from Phase 2 notebook:
  - ``[CLS]`` token contextual pooling
  - Two-layer classification head (Linear → GELU → Dropout → Linear)
  - ``LabelAttention`` mechanism (from ``AdvancedBertClassifier``)

**Modified**:
  - Default backbone switched from ``bert-base-uncased`` to PubMedBERT
  - Model returns **logits only** in ``forward()``; loss is computed
    externally by the training engine (clean separation per the plan).
  - ``LabelAttention`` is toggled via ``use_label_attention`` flag
  - Class renamed to ``ClinicalHMLTCModel``
"""

from __future__ import annotations

from typing import Dict, Optional

import torch
import torch.nn as nn
from transformers import AutoModel


# ════════════════════════════════════════════════════════════════════
# Label-Aware Attention  (kept from notebook 2 Section 5.2)
# ════════════════════════════════════════════════════════════════════

class LabelAttentionHead(nn.Module):
    """
    Label-aware attention mechanism.

    Each label has a learnable query vector that attends to the BERT
    sequence output, producing a label-specific representation.

    Kept from notebook 2 ``LabelAttention`` class.
    """

    def __init__(self, hidden_size: int, num_labels: int) -> None:
        super().__init__()
        self.label_queries = nn.Parameter(torch.randn(num_labels, hidden_size))
        self.attention_scale = hidden_size ** 0.5
        nn.init.xavier_uniform_(self.label_queries)

    def forward(
        self,
        hidden_states: torch.Tensor,
        attention_mask: Optional[torch.Tensor] = None,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        """
        Parameters
        ----------
        hidden_states : (batch, seq_len, hidden_size)
        attention_mask : (batch, seq_len)  — 1 for real tokens, 0 for padding

        Returns
        -------
        label_representations : (batch, num_labels, hidden_size)
        attn_weights          : (batch, num_labels, seq_len)
        """
        # Attention scores: (batch, num_labels, seq_len)
        scores = torch.matmul(
            self.label_queries.unsqueeze(0),      # (1, num_labels, hidden)
            hidden_states.transpose(1, 2),         # (batch, hidden, seq_len)
        ) / self.attention_scale

        # Mask padding tokens
        if attention_mask is not None:
            mask = attention_mask.unsqueeze(1).expand_as(scores)
            scores = scores.masked_fill(mask == 0, float("-inf"))

        attn_weights = torch.softmax(scores, dim=-1)
        label_representations = torch.matmul(attn_weights, hidden_states)

        return label_representations, attn_weights


# ════════════════════════════════════════════════════════════════════
# Main model
# ════════════════════════════════════════════════════════════════════

class ClinicalHMLTCModel(nn.Module):
    """
    PubMedBERT + multi-label classification head.

    Architecture (default — ``use_label_attention=False``)::

        Text → PubMedBERT → [CLS] → Dropout → FC → GELU → Dropout → FC → Logits

    Architecture (advanced — ``use_label_attention=True``)::

        Text → PubMedBERT → ┬─ [CLS] ──────────────────┐
                             └─ LabelAttention ──────────┤
                                                         ↓
                                                  Concat → FC → Logits

    Parameters
    ----------
    model_name : str
        HuggingFace model identifier.  Default is PubMedBERT.
    num_labels : int
        Number of output labels (parent + child, or flat).
    dropout_rate : float
    use_label_attention : bool
        If *True*, add the ``LabelAttentionHead`` from notebook V3 and
        fuse its output with the [CLS] representation.
    """

    def __init__(
        self,
        model_name: str = "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext",
        num_labels: int = 50,
        dropout_rate: float = 0.3,
        use_label_attention: bool = False,
    ) -> None:
        super().__init__()
        self.num_labels = num_labels
        self.use_label_attention = use_label_attention

        # ── Encoder ─────────────────────────────────────────────────
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size: int = self.encoder.config.hidden_size

        if use_label_attention:
            # ── V3 path: Label Attention + CLS fusion ───────────────
            self.label_attention = LabelAttentionHead(hidden_size, num_labels)
            self.dropout = nn.Dropout(dropout_rate)
            self.fusion = nn.Sequential(
                nn.Linear(hidden_size * 2, hidden_size),
                nn.GELU(),
                nn.Dropout(dropout_rate),
                nn.Linear(hidden_size, 1),
            )
        else:
            # ── V2 path: [CLS] + two-layer head ────────────────────
            self.classifier = nn.Sequential(
                nn.Dropout(dropout_rate),
                nn.Linear(hidden_size, hidden_size // 2),
                nn.GELU(),
                nn.Dropout(dropout_rate / 2),
                nn.Linear(hidden_size // 2, num_labels),
            )

    def forward(
        self,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
    ) -> torch.Tensor:
        """
        Forward pass.

        Returns
        -------
        logits : torch.Tensor of shape ``(batch, num_labels)``
            Raw logits (no sigmoid).  Loss is computed externally.
        """
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )
        sequence_output = outputs.last_hidden_state   # (B, L, H)
        cls_output = sequence_output[:, 0, :]          # (B, H)

        if self.use_label_attention:
            label_repr, _ = self.label_attention(sequence_output, attention_mask)
            # label_repr: (B, num_labels, H)
            cls_expanded = cls_output.unsqueeze(1).expand(-1, self.num_labels, -1)
            combined = torch.cat([cls_expanded, label_repr], dim=-1)
            combined = self.dropout(combined)
            logits = self.fusion(combined).squeeze(-1)   # (B, num_labels)
        else:
            logits = self.classifier(cls_output)          # (B, num_labels)

        return logits

    # ── Convenience ─────────────────────────────────────────────────

    def count_parameters(self) -> Dict[str, int]:
        total = sum(p.numel() for p in self.parameters())
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        return {"total": total, "trainable": trainable}

    def __repr__(self) -> str:
        params = self.count_parameters()
        return (
            f"ClinicalHMLTCModel("
            f"num_labels={self.num_labels}, "
            f"label_attention={self.use_label_attention}, "
            f"params={params['trainable']:,} trainable)"
        )


### ⚖️ Section 11 — Focal & Hierarchical Consistency Losses

Implements the advanced objective functions:

#### 1. Multi-Label Focal Loss
Combines Class-Balanced BCE with focusing weights to address extreme ICD label skew:

$$\mathcal{L}_{Focal} = -\alpha_t (1 - p_t)^\gamma \log(p_t)$$

#### 2. Hierarchical Consistency Loss
Penalizes violations where a child label probability exceeds its parent category probability:

$$\mathcal{L}_{H} = \frac{1}{|P|} \sum_{(c, p) \in P} \max\left(0, \hat{p}_c - \hat{p}_p\right)$$

#### 3. Combined Multi-Label Loss
Integrates classification targets with consistency constraints and label smoothing:

$$\mathcal{L}_{Total} = \mathcal{L}_{Focal} + \lambda \mathcal{L}_{H}$$

In [ ]:
"""
Configurable loss functions for hierarchical multi-label classification.

Extracted and operationalised from the scaffolded ``HierarchicalLoss``
in Phase 2 notebook Section 5.1.  Now split into composable, standalone
modules that can be used individually or combined:

  - ``FocalLoss``                — class-imbalance-aware BCE
  - ``HierarchicalConsistencyLoss`` — parent-child violation penalty
  - ``CombinedHMLTCLoss``       — composes any combination of the above

Usage
-----
Select via ``PipelineConfig.loss_type``:

    ============  ======================================
    loss_type     Result
    ============  ======================================
    ``"bce"``     Standard ``BCEWithLogitsLoss``
    ``"focal"``   ``FocalLoss``
    ``"hierarchical"``  ``BCEWithLogitsLoss`` + ``HierarchicalConsistencyLoss``
    ``"combined"``      ``FocalLoss`` + ``HierarchicalConsistencyLoss``
    ============  ======================================
"""

from __future__ import annotations

from typing import Dict, List, Optional, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F


# ════════════════════════════════════════════════════════════════════
# Focal Loss
# ════════════════════════════════════════════════════════════════════

class FocalLoss(nn.Module):
    """
    Focal loss for multi-label classification.

    Addresses class imbalance by down-weighting easy (well-classified)
    examples and focusing training on hard examples.

    Math (per element)::

        FL(p_t) = -alpha_t * (1 - p_t)^gamma * log(p_t)

    Kept from notebook 2 ``HierarchicalLoss.forward`` — the focal BCE
    computation — but extracted into a standalone module.

    Parameters
    ----------
    gamma : float
        Focusing parameter.  ``gamma=0`` reduces to standard BCE.
    alpha : float or None
        Balancing factor.  ``None`` = no alpha weighting.
    label_smoothing : float
        Smooth targets: ``y' = y*(1-eps) + eps/2``.
    reduction : str
        ``"mean"`` or ``"sum"`` or ``"none"``.
    """

    def __init__(
        self,
        gamma: float = 2.0,
        alpha: Optional[float] = None,
        label_smoothing: float = 0.0,
        reduction: str = "mean",
    ) -> None:
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha
        self.label_smoothing = label_smoothing
        self.reduction = reduction

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        """
        Parameters
        ----------
        logits  : (B, C)  raw logits
        targets : (B, C)  multi-hot targets  (0 or 1, float)
        """
        # Optional label smoothing
        if self.label_smoothing > 0:
            targets = targets * (1 - self.label_smoothing) + self.label_smoothing / 2

        # Element-wise BCE (no reduction)
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")

        probs = torch.sigmoid(logits)
        pt = targets * probs + (1 - targets) * (1 - probs)
        focal_weight = (1 - pt) ** self.gamma

        if self.alpha is not None:
            alpha_t = targets * self.alpha + (1 - targets) * (1 - self.alpha)
            focal_weight = alpha_t * focal_weight

        loss = focal_weight * bce

        if self.reduction == "mean":
            return loss.mean()
        elif self.reduction == "sum":
            return loss.sum()
        return loss


# ════════════════════════════════════════════════════════════════════
# Hierarchical Consistency Loss
# ════════════════════════════════════════════════════════════════════

class HierarchicalConsistencyLoss(nn.Module):
    """
    Penalises predictions where a child label is active but its parent
    is not.

    Kept from notebook 2 ``HierarchicalLoss`` — the hierarchy-penalty
    branch — extracted into a standalone module.

    Parameters
    ----------
    child_parent_pairs : list of (child_idx, parent_idx)
        Index pairs derived from the hierarchy.
    reduction : str
        ``"mean"`` or ``"sum"``.
    """

    def __init__(
        self,
        child_parent_pairs: List[Tuple[int, int]],
        reduction: str = "mean",
    ) -> None:
        super().__init__()
        self.child_parent_pairs = child_parent_pairs
        self.reduction = reduction

    @classmethod
    def from_hierarchy(
        cls,
        child_to_parent: Dict[str, str],
        label_to_idx: Dict[str, int],
        reduction: str = "mean",
    ) -> "HierarchicalConsistencyLoss":
        """
        Build from a ``child_to_parent`` mapping and ``label_to_idx`` dict.

        This is the recommended constructor when working with
        ``CodingSystemHierarchy`` and ``HierarchicalLabelEncoder``.
        """
        pairs: List[Tuple[int, int]] = []
        for child_label, parent_label in child_to_parent.items():
            if child_label in label_to_idx and parent_label in label_to_idx:
                pairs.append((label_to_idx[child_label], label_to_idx[parent_label]))
        return cls(pairs, reduction=reduction)

    def forward(self, logits: torch.Tensor) -> torch.Tensor:
        """
        Parameters
        ----------
        logits : (B, C)  raw logits

        Returns
        -------
        penalty : scalar tensor
        """
        if not self.child_parent_pairs:
            return torch.tensor(0.0, device=logits.device)

        probs = torch.sigmoid(logits)
        violations = []
        for child_idx, parent_idx in self.child_parent_pairs:
            # Penalise when child_prob > parent_prob
            violation = torch.relu(probs[:, child_idx] - probs[:, parent_idx])
            violations.append(violation)

        # Stack and reduce
        violations_tensor = torch.stack(violations, dim=1)  # (B, num_pairs)
        if self.reduction == "mean":
            return violations_tensor.mean()
        return violations_tensor.sum()


# ════════════════════════════════════════════════════════════════════
# Combined Loss (configurable composition)
# ════════════════════════════════════════════════════════════════════

class CombinedHMLTCLoss(nn.Module):
    """
    Compose primary classification loss + optional hierarchical penalty.

    Parameters
    ----------
    primary_loss : nn.Module
        The main classification loss (``FocalLoss`` or ``BCEWithLogitsLoss``).
    hierarchy_loss : HierarchicalConsistencyLoss or None
        Optional hierarchy-violation penalty.
    hierarchy_weight : float
        Weight for the hierarchy penalty term.
    """

    def __init__(
        self,
        primary_loss: nn.Module,
        hierarchy_loss: Optional[HierarchicalConsistencyLoss] = None,
        hierarchy_weight: float = 0.5,
    ) -> None:
        super().__init__()
        self.primary_loss = primary_loss
        self.hierarchy_loss = hierarchy_loss
        self.hierarchy_weight = hierarchy_weight

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        """
        Parameters
        ----------
        logits  : (B, C)
        targets : (B, C)
        """
        loss = self.primary_loss(logits, targets)

        if self.hierarchy_loss is not None and self.hierarchy_weight > 0:
            h_penalty = self.hierarchy_loss(logits)
            loss = loss + self.hierarchy_weight * h_penalty

        return loss


# ════════════════════════════════════════════════════════════════════
# Factory — build loss from config
# ════════════════════════════════════════════════════════════════════

def build_loss(
    loss_type: str,
    *,
    focal_gamma: float = 2.0,
    focal_alpha: Optional[float] = None,
    label_smoothing: float = 0.0,
    hierarchy_penalty_weight: float = 0.5,
    child_to_parent: Optional[Dict[str, str]] = None,
    label_to_idx: Optional[Dict[str, int]] = None,
) -> nn.Module:
    """
    Factory function: create the appropriate loss module from a string key.

    Parameters
    ----------
    loss_type : str
        One of ``"bce"``, ``"focal"``, ``"hierarchical"``, ``"combined"``.
    child_to_parent, label_to_idx
        Required when *loss_type* includes a hierarchy component.

    Returns
    -------
    nn.Module
    """
    # Primary loss
    if loss_type in ("bce", "hierarchical"):
        primary: nn.Module = nn.BCEWithLogitsLoss()
    elif loss_type in ("focal", "combined"):
        primary = FocalLoss(
            gamma=focal_gamma,
            alpha=focal_alpha,
            label_smoothing=label_smoothing,
        )
    else:
        raise ValueError(
            f"Unknown loss_type={loss_type!r}. "
            f"Choose from: 'bce', 'focal', 'hierarchical', 'combined'."
        )

    # Hierarchy penalty
    h_loss: Optional[HierarchicalConsistencyLoss] = None
    if loss_type in ("hierarchical", "combined"):
        if child_to_parent is None or label_to_idx is None:
            raise ValueError(
                f"loss_type={loss_type!r} requires 'child_to_parent' and "
                f"'label_to_idx' arguments to build the hierarchy penalty."
            )
        h_loss = HierarchicalConsistencyLoss.from_hierarchy(
            child_to_parent, label_to_idx
        )

    if h_loss is not None:
        return CombinedHMLTCLoss(primary, h_loss, hierarchy_penalty_weight)

    return primary


### 🚀 Section 12 — Training, Validation & Fit Engine

Provides clean, stateless functions (`train_one_epoch`, `validate`, `fit`). The `train_one_epoch` routine functions as the principal client-local training interface, completely decoupled from global model state.

In [ ]:
"""
Training engine — ``train_one_epoch``, ``validate``, ``fit``.

**Kept** from Phase 2 notebook:
  - Training loop structure (optimizer, scheduler, gradient accumulation,
    gradient clipping, best-model tracking via ``deepcopy``)
  - ``evaluate_transformer`` → renamed ``validate``

**Modified**:
  - Extracted into clean, stateless functions (no ``Config`` dependency —
    all parameters are passed as arguments).
  - ``train_one_epoch`` is the **key FL handoff function**: it takes model,
    dataloader, optimizer, loss_fn, device and returns ``(avg_loss, metrics)``.
  - ``fit`` composes ``train_one_epoch`` + ``validate`` for centralized
    baseline experiments.
"""

from __future__ import annotations

from copy import deepcopy
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import f1_score
from torch.utils.data import DataLoader
from tqdm import tqdm


# ════════════════════════════════════════════════════════════════════
# Single-epoch training  (FL handoff function)
# ════════════════════════════════════════════════════════════════════

def train_one_epoch(
    model: nn.Module,
    dataloader: DataLoader,
    optimizer: torch.optim.Optimizer,
    loss_fn: nn.Module,
    device: str | torch.device,
    *,
    scheduler: Optional[Any] = None,
    grad_accum_steps: int = 1,
    max_grad_norm: float = 1.0,
    epoch_idx: int = 0,
    total_epochs: int = 1,
    silent: bool = False,
) -> Dict[str, float]:
    """
    Run **one** local training epoch.

    This is the function that the FL team calls per federated round.

    Parameters
    ----------
    model : nn.Module
        The ``ClinicalHMLTCModel`` (must already be on *device*).
    dataloader : DataLoader
        Training data for this epoch.
    optimizer : Optimizer
    loss_fn : nn.Module
        Any loss from ``nlp_pipeline.models.loss``.
    device : str or torch.device
    scheduler : optional LR scheduler (stepped per optimizer step)
    grad_accum_steps : int
    max_grad_norm : float
    epoch_idx, total_epochs : ints  (for progress-bar display)
    silent : bool  — suppress progress bar

    Returns
    -------
    dict with keys ``{"train_loss": float, "num_samples": int}``
    """
    model.train()
    epoch_loss = 0.0
    num_samples = 0
    optimizer.zero_grad()

    iterator = dataloader
    if not silent:
        iterator = tqdm(dataloader, desc=f"Epoch {epoch_idx+1}/{total_epochs} [Train]")

    for step, batch in enumerate(iterator):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        logits = model(input_ids, attention_mask)
        loss = loss_fn(logits, labels) / grad_accum_steps
        loss.backward()

        batch_loss = loss.item() * grad_accum_steps
        num_samples += input_ids.size(0)

        if (step + 1) % grad_accum_steps == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=max_grad_norm)
            optimizer.step()
            if scheduler is not None:
                scheduler.step()
            optimizer.zero_grad()

        epoch_loss += batch_loss

        if not silent:
            iterator.set_postfix({"loss": f"{batch_loss:.4f}"})

    # Handle remaining gradients if steps not divisible by accum
    if (step + 1) % grad_accum_steps != 0:
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=max_grad_norm)
        optimizer.step()
        if scheduler is not None:
            scheduler.step()
        optimizer.zero_grad()

    avg_loss = epoch_loss / len(dataloader)
    return {"train_loss": avg_loss, "num_samples": num_samples}


# ════════════════════════════════════════════════════════════════════
# Validation / evaluation pass
# ════════════════════════════════════════════════════════════════════

def validate(
    model: nn.Module,
    dataloader: DataLoader,
    loss_fn: nn.Module,
    device: str | torch.device,
    *,
    threshold: float = 0.5,
) -> Tuple[float, np.ndarray, np.ndarray]:
    """
    Evaluate the model on a DataLoader **without** gradient computation.

    Kept from notebook 2 ``evaluate_transformer()``.

    Returns
    -------
    avg_loss : float
    predictions : np.ndarray  (N, C)  binary
    true_labels : np.ndarray  (N, C)  binary
    """
    model.eval()
    total_loss = 0.0
    all_preds: List[np.ndarray] = []
    all_labels: List[np.ndarray] = []

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            logits = model(input_ids, attention_mask)
            loss = loss_fn(logits, labels)
            total_loss += loss.item()

            probs = torch.sigmoid(logits)
            preds = (probs >= threshold).int().cpu().numpy()
            all_preds.append(preds)
            all_labels.append(labels.cpu().numpy().astype(int))

    avg_loss = total_loss / max(len(dataloader), 1)
    return avg_loss, np.vstack(all_preds), np.vstack(all_labels)


# ════════════════════════════════════════════════════════════════════
# Full training loop (centralized baseline)
# ════════════════════════════════════════════════════════════════════

def fit(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    loss_fn: nn.Module,
    device: str | torch.device,
    *,
    num_epochs: int = 5,
    scheduler: Optional[Any] = None,
    grad_accum_steps: int = 1,
    max_grad_norm: float = 1.0,
    threshold: float = 0.5,
) -> Dict[str, List[float]]:
    """
    Full centralized training loop with validation monitoring.

    Composes ``train_one_epoch`` + ``validate`` and tracks the best
    model by validation F1 micro.

    Returns
    -------
    history : dict with keys
        ``train_loss``, ``val_loss``, ``val_f1_micro``, ``val_f1_macro``
    """
    model.to(device)

    history: Dict[str, List[float]] = {
        "train_loss": [],
        "val_loss": [],
        "val_f1_micro": [],
        "val_f1_macro": [],
    }
    best_val_f1 = 0.0
    best_model_state = None

    for epoch in range(num_epochs):
        # ── Train ───────────────────────────────────────────────
        train_metrics = train_one_epoch(
            model, train_loader, optimizer, loss_fn, device,
            scheduler=scheduler,
            grad_accum_steps=grad_accum_steps,
            max_grad_norm=max_grad_norm,
            epoch_idx=epoch,
            total_epochs=num_epochs,
        )
        history["train_loss"].append(train_metrics["train_loss"])

        # ── Validate ────────────────────────────────────────────
        val_loss, val_preds, val_labels = validate(
            model, val_loader, loss_fn, device, threshold=threshold,
        )
        history["val_loss"].append(val_loss)

        val_f1_micro = float(f1_score(val_labels, val_preds, average="micro", zero_division=0))
        val_f1_macro = float(f1_score(val_labels, val_preds, average="macro", zero_division=0))
        history["val_f1_micro"].append(val_f1_micro)
        history["val_f1_macro"].append(val_f1_macro)

        print(
            f"  Epoch {epoch+1}: train_loss={train_metrics['train_loss']:.4f} | "
            f"val_loss={val_loss:.4f} | val_f1_micro={val_f1_micro:.4f} | "
            f"val_f1_macro={val_f1_macro:.4f}"
        )

        # ── Best model tracking ─────────────────────────────────
        if val_f1_micro > best_val_f1:
            best_val_f1 = val_f1_micro
            best_model_state = deepcopy(model.state_dict())
            print(f"    -> New best model (F1 micro: {best_val_f1:.4f})")

    # Restore best model
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
        print(f"\n✓ Restored best model (val F1 micro: {best_val_f1:.4f})")

    return history


### 📊 Section 13 — Multi-Label Evaluation Metrics

Tracks standard clinical evaluation metrics: Subset Accuracy, Jaccard Index, Hamming Loss, and Micro/Macro Precision, Recall, and F1. Also checks hierarchical consistency rates across predicted classes.

In [ ]:
"""
Evaluation metrics for hierarchical multi-label classification.

**Kept verbatim** from Phase 2 notebook:
  - ``evaluate_multilabel``            — full metric suite
  - ``check_hierarchical_consistency`` — hierarchy violation analysis
"""

from __future__ import annotations

from collections import Counter
from typing import Any, Dict, List, Optional

import numpy as np
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    hamming_loss,
    jaccard_score,
    precision_score,
    recall_score,
)

from nlp_pipeline.utils.helpers import timestamp


def evaluate_multilabel(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    label_names: List[str],
    model_name: str = "Model",
    *,
    print_report: bool = True,
) -> Dict[str, Any]:
    """
    Comprehensive multi-label evaluation.

    Returns a dict of metrics and optionally prints a detailed report.

    Kept from notebook 2 ``evaluate_multilabel()``.

    Parameters
    ----------
    y_true : np.ndarray of shape (N, C)
    y_pred : np.ndarray of shape (N, C)
    label_names : list[str]
    model_name : str
    print_report : bool
    """
    metrics: Dict[str, Any] = {
        "model": model_name,
        "timestamp": timestamp(),
        "hamming_loss": float(hamming_loss(y_true, y_pred)),
        "subset_accuracy": float(accuracy_score(y_true, y_pred)),
        "f1_micro": float(f1_score(y_true, y_pred, average="micro", zero_division=0)),
        "f1_macro": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "f1_weighted": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
        "f1_samples": float(f1_score(y_true, y_pred, average="samples", zero_division=0)),
        "precision_micro": float(precision_score(y_true, y_pred, average="micro", zero_division=0)),
        "recall_micro": float(recall_score(y_true, y_pred, average="micro", zero_division=0)),
        "precision_macro": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "recall_macro": float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "jaccard_micro": float(jaccard_score(y_true, y_pred, average="micro", zero_division=0)),
        "jaccard_macro": float(jaccard_score(y_true, y_pred, average="macro", zero_division=0)),
    }

    # Per-label metrics
    per_label: Dict[str, Dict[str, float]] = {}
    for i, name in enumerate(label_names):
        support = int(y_true[:, i].sum())
        if support > 0:
            per_label[name] = {
                "precision": float(precision_score(y_true[:, i], y_pred[:, i], zero_division=0)),
                "recall": float(recall_score(y_true[:, i], y_pred[:, i], zero_division=0)),
                "f1": float(f1_score(y_true[:, i], y_pred[:, i], zero_division=0)),
                "support": support,
            }
    metrics["per_label"] = per_label

    # Print report
    if print_report:
        print(f"\n{'=' * 65}")
        print(f"  EVALUATION REPORT: {model_name}")
        print(f"{'=' * 65}")
        print(f"  Hamming Loss      : {metrics['hamming_loss']:.4f}")
        print(f"  Subset Accuracy   : {metrics['subset_accuracy']:.4f}")
        print(f"  F1 Micro          : {metrics['f1_micro']:.4f}")
        print(f"  F1 Macro          : {metrics['f1_macro']:.4f}")
        print(f"  F1 Weighted       : {metrics['f1_weighted']:.4f}")
        print(f"  F1 Samples        : {metrics['f1_samples']:.4f}")
        print(f"  Precision (micro) : {metrics['precision_micro']:.4f}")
        print(f"  Recall (micro)    : {metrics['recall_micro']:.4f}")
        print(f"  Jaccard (micro)   : {metrics['jaccard_micro']:.4f}")
        print(f"{'=' * 65}")

        print(f"\n  {'Label':<30} {'Prec':>8} {'Recall':>8} {'F1':>8} {'Support':>8}")
        print(f"  {'-' * 62}")
        for name, vals in sorted(per_label.items()):
            print(
                f"  {name:<30} {vals['precision']:>8.3f} {vals['recall']:>8.3f} "
                f"{vals['f1']:>8.3f} {vals['support']:>8}"
            )
        print(f"  {'-' * 62}")

    return metrics


def check_hierarchical_consistency(
    y_pred: np.ndarray,
    label_names: List[str],
    child_to_parent: Dict[str, str],
) -> Dict[str, Any]:
    """
    Check how often predictions violate the label hierarchy
    (child predicted without parent).

    Kept from notebook 2 ``check_hierarchical_consistency()``.

    Parameters
    ----------
    y_pred : np.ndarray  (N, C)
    label_names : list[str]
    child_to_parent : dict  mapping child→parent label strings
    """
    total_preds = 0
    violations = 0
    violation_details: Counter = Counter()

    for sample in y_pred:
        active_labels = set(
            label_names[i] for i in range(len(label_names)) if sample[i] == 1
        )
        for child_label in active_labels:
            if child_label in child_to_parent:
                parent_label = child_to_parent[child_label]
                total_preds += 1
                if parent_label not in active_labels:
                    violations += 1
                    violation_details[f"{child_label} without {parent_label}"] += 1

    consistency_rate = 1 - (violations / max(total_preds, 1))
    return {
        "total_child_predictions": total_preds,
        "violations": violations,
        "consistency_rate": consistency_rate,
        "top_violations": dict(violation_details.most_common(10)),
    }


### 🔌 Section 14 — Public NLP API & Federated Handoff Client

Defines the integration boundary for the FL simulation team: **`FederatedNLPClient`**.

> [!NOTE]
> Decouples model weights and optimizer instances from client orchestration. Exposes the exact ports required for weight transmission and execution:
- `get_weights()` / `set_weights(state)`
- `local_train_epoch(loader)`
- `get_num_samples()`
- `evaluate(loader)`

In [ ]:
"""
Public NLP API and Federated Learning client interface.

This module is the **single integration surface** between the NLP pipeline
and the Federated Learning framework.  The FL team (Ratan & Harshit) should
only need to interact with ``FederatedNLPClient``.

Design decisions
----------------
- The client exposes **only** the APIs required for local training and
  model-weight exchange: ``get_weights``, ``set_weights``, ``local_train_epoch``,
  ``evaluate``, ``get_num_samples``.
- All FL-specific optimization (FedAvg weighting, FedProx μ, SCAFFOLD
  corrections, differential-privacy noise injection) is **not** handled
  here.  Those are the FL team's responsibility and can be applied
  externally around the ``get_weights`` / ``set_weights`` boundary.
- The client wraps ``ClinicalHMLTCModel`` + ``CombinedHMLTCLoss`` +
  ``AdamW`` optimizer internally so the FL team doesn't need to
  understand the model internals.

Standalone convenience functions (``initialize_model``, ``preprocess``,
``build_dataloaders``, etc.) are also provided for centralized experiments
and scripting.
"""

from __future__ import annotations

from collections import OrderedDict
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, get_linear_schedule_with_warmup

from nlp_pipeline.configs.config import PipelineConfig
from nlp_pipeline.data.data_plugin import BaseDataPlugin
from nlp_pipeline.data.dataset import ClinicalTextDataset
from nlp_pipeline.data.hierarchy import CodingSystemHierarchy
from nlp_pipeline.data.label_encoder import HierarchicalLabelEncoder
from nlp_pipeline.data.preprocess import clean_clinical_text
from nlp_pipeline.models.loss import build_loss
from nlp_pipeline.models.model import ClinicalHMLTCModel
from nlp_pipeline.training.metrics import evaluate_multilabel
from nlp_pipeline.training.trainer import train_one_epoch, validate


# ════════════════════════════════════════════════════════════════════
# Federated NLP Client
# ════════════════════════════════════════════════════════════════════

class FederatedNLPClient:
    """
    Encapsulates the NLP model, loss, optimizer, and tokenizer for one
    federated client.  Exposes the exact handoff ports the FL team needs.

    Usage (FL side)::

        client = FederatedNLPClient(config, hierarchy, label_encoder)
        client.set_weights(global_state_dict)        # receive global model
        metrics = client.local_train_epoch(loader)    # one local epoch
        updated_weights = client.get_weights()        # send back to server
        n = client.get_num_samples()                  # for FedAvg weighting

    Parameters
    ----------
    config : PipelineConfig
    hierarchy : CodingSystemHierarchy
    label_encoder : HierarchicalLabelEncoder
    """

    def __init__(
        self,
        config: PipelineConfig,
        hierarchy: CodingSystemHierarchy,
        label_encoder: HierarchicalLabelEncoder,
    ) -> None:
        self.config = config
        self.hierarchy = hierarchy
        self.label_encoder = label_encoder
        self.device = torch.device(config.device)

        # ── Model ───────────────────────────────────────────────
        self.model = ClinicalHMLTCModel(
            model_name=config.model_name,
            num_labels=label_encoder.num_labels,
            dropout_rate=config.dropout_rate,
            use_label_attention=config.use_label_attention,
        ).to(self.device)

        # ── Loss ────────────────────────────────────────────────
        self.loss_fn: nn.Module = build_loss(
            loss_type=config.loss_type,
            focal_gamma=config.focal_gamma,
            focal_alpha=config.focal_alpha,
            label_smoothing=config.label_smoothing,
            hierarchy_penalty_weight=config.hierarchy_penalty_weight,
            child_to_parent=hierarchy.child_to_parent,
            label_to_idx=label_encoder.label_to_idx,
        )

        # ── Tokenizer ──────────────────────────────────────────
        self.tokenizer = AutoTokenizer.from_pretrained(config.model_name)

        # ── Optimizer (created lazily or on first train) ───────
        self._optimizer: Optional[AdamW] = None
        self._scheduler: Optional[Any] = None
        self._num_samples: int = 0

    # ═══════════════════════════════════════════════════════════
    #  FL Handoff Ports
    # ═══════════════════════════════════════════════════════════

    def get_weights(self) -> OrderedDict:
        """
        Export model ``state_dict`` for aggregation.

        The FL server calls this after ``local_train_epoch`` to retrieve
        the updated weights.
        """
        return OrderedDict(
            (k, v.cpu().clone()) for k, v in self.model.state_dict().items()
        )

    def set_weights(self, state_dict: OrderedDict) -> None:
        """
        Load global model weights received from the FL server.

        Call this **before** ``local_train_epoch`` each round.
        """
        self.model.load_state_dict(state_dict, strict=True)
        self.model.to(self.device)
        # Reset optimizer so momentum buffers align with new weights
        self._optimizer = None
        self._scheduler = None

    def local_train_epoch(
        self,
        dataloader: DataLoader,
        **kwargs: Any,
    ) -> Dict[str, float]:
        """
        Run **one** local training epoch.

        This is the core function the FL framework calls per round.

        Parameters
        ----------
        dataloader : DataLoader
            Client-local training data.
        **kwargs
            Forwarded to ``train_one_epoch`` (e.g. ``silent=True``).

        Returns
        -------
        dict with ``{"train_loss": float, "num_samples": int}``
        """
        # Lazy-init optimizer
        if self._optimizer is None:
            self._optimizer = AdamW(
                self.model.parameters(),
                lr=self.config.learning_rate,
                weight_decay=self.config.weight_decay,
            )

        self._num_samples = len(dataloader.dataset)

        metrics = train_one_epoch(
            model=self.model,
            dataloader=dataloader,
            optimizer=self._optimizer,
            loss_fn=self.loss_fn,
            device=self.device,
            grad_accum_steps=self.config.grad_accum_steps,
            max_grad_norm=self.config.max_grad_norm,
            **kwargs,
        )
        return metrics

    def get_num_samples(self) -> int:
        """
        Return the number of samples in the client's training set.

        Needed by FedAvg for proportional aggregation weighting.
        """
        return self._num_samples

    # ═══════════════════════════════════════════════════════════
    #  Evaluation & Inference
    # ═══════════════════════════════════════════════════════════

    def evaluate(
        self,
        dataloader: DataLoader,
        model_name: str = "FederatedClient",
    ) -> Dict[str, Any]:
        """
        Evaluate the model on a DataLoader and return the full metric dict.

        Parameters
        ----------
        dataloader : DataLoader
        model_name : str   for the evaluation report header

        Returns
        -------
        dict   (same structure as ``evaluate_multilabel``)
        """
        val_loss, preds, labels = validate(
            self.model, dataloader, self.loss_fn, self.device,
            threshold=self.config.classification_threshold,
        )
        metrics = evaluate_multilabel(
            labels, preds, self.label_encoder.label_names,
            model_name=model_name, print_report=False,
        )
        metrics["val_loss"] = val_loss
        return metrics

    def predict(
        self,
        texts: List[str],
        batch_size: int = 16,
    ) -> np.ndarray:
        """
        Run inference on raw text inputs.

        Parameters
        ----------
        texts : list[str]
        batch_size : int

        Returns
        -------
        np.ndarray  (N, num_labels)  — predicted probabilities
        """
        self.model.eval()
        all_probs: List[np.ndarray] = []

        for i in range(0, len(texts), batch_size):
            batch_texts = texts[i : i + batch_size]
            encoding = self.tokenizer(
                batch_texts,
                max_length=self.config.max_seq_length,
                padding="max_length",
                truncation=True,
                return_tensors="pt",
            )
            input_ids = encoding["input_ids"].to(self.device)
            attention_mask = encoding["attention_mask"].to(self.device)

            with torch.no_grad():
                logits = self.model(input_ids, attention_mask)
                probs = torch.sigmoid(logits).cpu().numpy()
                all_probs.append(probs)

        return np.vstack(all_probs)

    def predict_labels(
        self,
        texts: List[str],
        batch_size: int = 16,
    ) -> List[List[str]]:
        """
        Predict and decode to label strings.

        Returns
        -------
        list[list[str]]  — decoded label lists per sample
        """
        probs = self.predict(texts, batch_size)
        return self.label_encoder.decode(probs, self.config.classification_threshold)

    # ═══════════════════════════════════════════════════════════
    #  DataLoader Construction
    # ═══════════════════════════════════════════════════════════

    def build_dataloader(
        self,
        records: List[Dict[str, Any]],
        shuffle: bool = True,
        batch_size: Optional[int] = None,
    ) -> DataLoader:
        """
        Build a ``DataLoader`` from standardised records.

        Parameters
        ----------
        records : list[dict]
            Each dict must have ``"text"`` and ``"labels"`` keys.
        shuffle : bool
        batch_size : int or None  (defaults to ``config.batch_size``)
        """
        dataset = ClinicalTextDataset.from_records(
            records=records,
            label_encoder=self.label_encoder,
            tokenizer=self.tokenizer,
            max_length=self.config.max_seq_length,
        )
        return DataLoader(
            dataset,
            batch_size=batch_size or self.config.batch_size,
            shuffle=shuffle,
            num_workers=self.config.num_workers,
            pin_memory=self.config.pin_memory,
        )

    # ═══════════════════════════════════════════════════════════
    #  Checkpoint helpers
    # ═══════════════════════════════════════════════════════════

    def save_model(self, filepath: str) -> None:
        """Save model state_dict to disk."""
        torch.save(self.model.state_dict(), filepath)
        print(f"  ✓ Model saved: {filepath}")

    def load_model(self, filepath: str) -> None:
        """Load model state_dict from disk."""
        state = torch.load(filepath, map_location=self.device, weights_only=True)
        self.model.load_state_dict(state)
        print(f"  ✓ Model loaded: {filepath}")

    def __repr__(self) -> str:
        return (
            f"FederatedNLPClient("
            f"model={self.config.model_name!r}, "
            f"labels={self.label_encoder.num_labels}, "
            f"device={self.device})"
        )


# ════════════════════════════════════════════════════════════════════
# Standalone convenience functions (for centralized experiments)
# ════════════════════════════════════════════════════════════════════

def initialize_model(config: PipelineConfig, num_labels: int) -> ClinicalHMLTCModel:
    """Create a ``ClinicalHMLTCModel`` from config."""
    return ClinicalHMLTCModel(
        model_name=config.model_name,
        num_labels=num_labels,
        dropout_rate=config.dropout_rate,
        use_label_attention=config.use_label_attention,
    )


def preprocess(records: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """Clean text in a list of standardised records."""
    return [
        {**r, "text": clean_clinical_text(r.get("text", ""))}
        for r in records
    ]


def build_dataloaders(
    train_records: List[Dict],
    val_records: List[Dict],
    test_records: List[Dict],
    label_encoder: HierarchicalLabelEncoder,
    tokenizer: Any,
    config: PipelineConfig,
) -> Tuple[DataLoader, DataLoader, DataLoader]:
    """Build train / val / test DataLoaders from record lists."""
    def _make(records: List[Dict], shuffle: bool) -> DataLoader:
        ds = ClinicalTextDataset.from_records(
            records, label_encoder, tokenizer, config.max_seq_length,
        )
        return DataLoader(
            ds,
            batch_size=config.batch_size,
            shuffle=shuffle,
            num_workers=config.num_workers,
            pin_memory=config.pin_memory,
        )

    return _make(train_records, True), _make(val_records, False), _make(test_records, False)


### 🧪 Section 15 — Empirical Verification & Unit-Level Tests

This section executes integration checks to verify standard operations: coding-system hierarchy mapping, label encoding expansion, losses scaling, and the `FederatedNLPClient` weight round-trip saving/loading.

In [ ]:
"""
Functional verification script for the nlp_pipeline package.
Tests hierarchy, label encoder, loss factory, and FederatedNLPClient round-trip.
"""
import sys
import torch
import numpy as np

# ── 1. Hierarchy from ICD-9 codes ────────────────────────────────
from nlp_pipeline.data.hierarchy import CodingSystemHierarchy

icd9_codes = ["428.0", "428.32", "410.9", "250.00", "250.01", "786.05", "401.9"]
h = CodingSystemHierarchy.from_codes(icd9_codes, system="icd9", depth=3)
print(h.summary())
print()
assert h.num_parents > 0, "No parents built"
assert h.num_children > 0, "No children built"
assert h.coding_system == "icd9"
print("✓ Hierarchy from ICD-9 codes")

# ── 2. Hierarchy from custom mapping ────────────────────────────
custom = CodingSystemHierarchy.from_mapping({
    "Circulatory": ["HF", "MI"],
    "Endocrine": ["DM"],
})
assert custom.num_labels == 5  # 2 parents + 3 children
assert custom.get_parent("HF") == "Circulatory"
print("✓ Hierarchy from custom mapping")

# ── 3. Auto-detect ICD-10 codes ──────────────────────────────────
icd10_codes = ["I50.9", "E11.65", "J44.1"]
h10 = CodingSystemHierarchy.from_codes(icd10_codes, system="auto")
assert h10.coding_system == "icd10"
print("✓ Auto-detected ICD-10")

# ── 4. Label encoder ────────────────────────────────────────────
from nlp_pipeline.data.label_encoder import HierarchicalLabelEncoder

encoder = HierarchicalLabelEncoder(custom)
assert encoder.num_labels == 5

# Encode with parent consistency
vec = encoder.encode_single(["HF"])  # Should auto-add "Circulatory"
assert vec[encoder.label_to_idx["Circulatory"]] == 1.0, "Parent not auto-added"
assert vec[encoder.label_to_idx["HF"]] == 1.0
print("✓ Label encoder with parent consistency")

# Batch encode + decode round-trip
labels_batch = [["HF", "MI"], ["DM"]]
matrix = encoder.encode(labels_batch)
assert matrix.shape == (2, 5)
decoded = encoder.decode(matrix, threshold=0.5)
assert "HF" in decoded[0] and "MI" in decoded[0]
assert "DM" in decoded[1]
print("✓ Encode/decode round-trip")

# ── 5. Loss factory ─────────────────────────────────────────────
from nlp_pipeline.models.loss import build_loss

for lt in ["bce", "focal", "hierarchical", "combined"]:
    kwargs = {}
    if lt in ("hierarchical", "combined"):
        kwargs["child_to_parent"] = custom.child_to_parent
        kwargs["label_to_idx"] = encoder.label_to_idx
    loss_fn = build_loss(lt, **kwargs)
    # Quick forward pass
    logits = torch.randn(4, 5)
    targets = torch.randint(0, 2, (4, 5)).float()
    loss_val = loss_fn(logits, targets)
    assert loss_val.dim() == 0, f"Loss {lt} not scalar"
    print(f"✓ Loss '{lt}' forward OK — value={loss_val.item():.4f}")

# ── 6. FederatedNLPClient get/set weights round-trip ─────────────
from nlp_pipeline.configs.config import PipelineConfig
from nlp_pipeline.api.interface import FederatedNLPClient

# Use a tiny model for speed
config = PipelineConfig(
    model_name="google/bert_uncased_L-2_H-128_A-2",  # tiny BERT for test
    loss_type="bce",
    batch_size=2,
    num_epochs=1,
)

client = FederatedNLPClient(config, custom, encoder)
print(f"\n{client}")

# Round-trip weights
w1 = client.get_weights()
client.set_weights(w1)
w2 = client.get_weights()

for key in w1:
    assert torch.equal(w1[key], w2[key]), f"Mismatch at {key}"
print("✓ get_weights / set_weights round-trip")

# ── 7. Predict on dummy text ─────────────────────────────────────
probs = client.predict(["Heart failure with reduced ejection fraction."])
assert probs.shape == (1, 5)
print(f"✓ Predict: probabilities shape {probs.shape}")

labels_out = client.predict_labels(["Patient has diabetes mellitus type II."])
print(f"✓ predict_labels: {labels_out}")

print("\n" + "=" * 60)
print("ALL FUNCTIONAL TESTS PASSED")
print("=" * 60)
